# Phase 16 — Human Validation Labels and Label Governance

This notebook creates the evaluation-only manual label governance workflow for `pairs_v2`. It samples balanced validation/test cases, publishes reviewer guidelines, validates collected labels, reports reviewer agreement, and freezes immutable manual-label artifacts only when human reviewer labels are present.

Manual labels are never joined back into training features. They are evaluation-only evidence for later baseline, model, calibration, and model-card phases.

## Purpose
Document and verify Phase 16 — Human Validation Labels and Label Governance in the Bisakerja notebook-first training workflow.

## Required input
Use the repository-root training data, artifacts, and reports referenced by this phase.

## Action
Run or review the Phase 16.human.validation.label.governance notebook cells in numeric order, preserving generated evidence under reports/ and artifacts/.

## Expected output
Produce or preserve the phase-specific report and artifact evidence for Phase 16 — Human Validation Labels and Label Governance.

## Verification
Confirm the notebook has no saved error outputs, no unintended unexecuted production code cells, and matching durable report evidence.

## Step 16.1 — Reviewer guidelines

### Purpose
Create durable reviewer instructions for job-fit score bands, ATS issue labels, recommendation relevance, and unsupported-claim rejection.

### Required input
Phase 15 `pairs_v2` score bands, model-core output boundaries, and reviewer governance config.

### Action
Write English reviewer guidelines to `reports/phase_16_reviewer_guidelines.md` and store guideline metadata in the phase report.

### Expected output
A standalone reviewer guideline document with scoring rules, evidence-note requirements, disagreement flags, and prohibited claims.

### Verification
The guidelines file exists, has a SHA-256 hash, and covers all required governance topics.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

PHASE_ID = "phase_16_human_validation_label_governance"
SCHEMA_VERSION = "human-validation-label-governance-v1"
LABEL_VERSION = "human-validation-v1"
RANDOM_SEED = 202616

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "TODOS.md").exists():
    for parent in Path.cwd().parents:
        if (parent / "TODOS.md").exists():
            REPO_ROOT = parent
            break

REPORTS_DIR = REPO_ROOT / "reports"
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
MANUAL_LABEL_DIR = ARTIFACTS_DIR / "manual_validation"
PAIRS_PATH = ARTIFACTS_DIR / "pairs_v2.parquet"
GUIDELINES_PATH = REPORTS_DIR / "phase_16_reviewer_guidelines.md"
REVIEW_QUEUE_PATH = MANUAL_LABEL_DIR / "phase_16_review_queue.csv"
LABEL_TEMPLATE_PATH = MANUAL_LABEL_DIR / "phase_16_human_label_template.csv"
HUMAN_LABELS_PATH = MANUAL_LABEL_DIR / "phase_16_human_labels.csv"
FROZEN_LABELS_PATH = MANUAL_LABEL_DIR / "phase_16_human_labels_frozen.csv"
LABEL_MANIFEST_PATH = REPORTS_DIR / "phase_16_label_manifest.json"
PHASE_REPORT_PATH = REPORTS_DIR / "phase_16_human_validation_label_governance.json"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MANUAL_LABEL_DIR.mkdir(parents=True, exist_ok=True)
random.seed(RANDOM_SEED)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def score_band_from_value(value: float) -> str:
    if value >= 70:
        return "high"
    if value >= 40:
        return "medium"
    return "low"


def normalize_label_score(value: Any) -> float | None:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    text = str(value).strip()
    if not text:
        return None
    try:
        score = float(text)
    except ValueError:
        return None
    if not 0 <= score <= 100:
        return None
    return score


def quadratic_weighted_kappa(labels_a: list[str], labels_b: list[str], ordered_labels: list[str]) -> float | None:
    if len(labels_a) != len(labels_b) or not labels_a:
        return None
    n = len(ordered_labels)
    index = {label: i for i, label in enumerate(ordered_labels)}
    observed = [[0.0 for _ in range(n)] for _ in range(n)]
    hist_a = [0.0 for _ in range(n)]
    hist_b = [0.0 for _ in range(n)]
    for a, b in zip(labels_a, labels_b):
        if a not in index or b not in index:
            continue
        i, j = index[a], index[b]
        observed[i][j] += 1.0
        hist_a[i] += 1.0
        hist_b[j] += 1.0
    total = sum(sum(row) for row in observed)
    if total == 0:
        return None
    weighted_observed = 0.0
    weighted_expected = 0.0
    for i in range(n):
        for j in range(n):
            weight = ((i - j) ** 2) / ((n - 1) ** 2) if n > 1 else 0.0
            weighted_observed += weight * observed[i][j] / total
            weighted_expected += weight * (hist_a[i] * hist_b[j]) / (total * total)
    if weighted_expected == 0:
        return 1.0 if weighted_observed == 0 else 0.0
    return 1.0 - (weighted_observed / weighted_expected)

GUIDELINES = """# Phase 16 Reviewer Guidelines — Human Validation Labels

## Purpose
Create trusted evaluation-only labels for job-fit, ATS quality, recommendation relevance, and unsupported-claim rejection. These labels measure model quality. They must never become model input features.

## Required evidence
Reviewers must inspect the sampled pair fields, matched/missing skills, role family, experience band, and any available CV/job text evidence. Every label needs an evidence note that cites observed facts.

## Job-fit score bands
- `low` (`0-39`): weak role/skill/requirement match, critical missing requirements, or clear seniority mismatch.
- `medium` (`40-69`): partial match with useful overlap but important gaps remain.
- `high` (`70-100`): strong role, skill, requirement, and seniority alignment with only minor gaps.

Reviewers may assign a numeric `reviewer_job_fit_score` from `0` to `100`; the score band is derived from that value.

## ATS issue labels
Mark issue flags only when evidence is present:
- `parseability_issue`: CV text is empty, garbled, scanned-only, or materially incomplete.
- `section_completeness_issue`: core sections such as experience, education, skills, or contact are missing.
- `contact_detection_issue`: contact details are absent or unreadable.
- `date_detection_issue`: experience or education dates are absent, contradictory, or unreadable.
- `metric_evidence_issue`: achievements lack measurable impact where role expectations require evidence.
- `formatting_risk_issue`: tables, columns, graphics, or unusual ordering create parser risk.

## Recommendation relevance
Use `low`, `medium`, or `high` relevance for whether the job should appear in a candidate recommendation set for this profile/CV.

## Unsupported-claim rejection
Set `unsupported_claim_flag=true` when generated or proposed output claims a skill, seniority, hiring likelihood, language ability, credential, or experience not supported by evidence.

## Disagreement flags
Set `disagreement_flag=true` when reviewer confidence is low, another reviewer should adjudicate, evidence is insufficient, or labels conflict with the score band.

## Governance rules
- Keep human labels evaluation-only.
- Do not edit frozen label files; create a new label version instead.
- Do not expose raw reviewer notes to user-facing product copy.
- Do not use manual labels to construct profile/job features, prompts, embeddings, or training targets unless a future explicitly approved label-training phase creates a separate training dataset.
"""

GUIDELINES_PATH.write_text(GUIDELINES, encoding="utf-8")
guidelines_record = {
    "path": str(GUIDELINES_PATH.relative_to(REPO_ROOT)),
    "sha256": sha256_file(GUIDELINES_PATH),
    "covers": [
        "job_fit_score_bands",
        "ats_issue_labels",
        "recommendation_relevance",
        "unsupported_claim_rejection",
        "evidence_notes",
        "disagreement_flags",
        "evaluation_only_policy",
    ],
}

guidelines_record


{'path': 'reports/phase_16_reviewer_guidelines.md',
 'sha256': 'c0cc41922f7f779c01a041bfa5926cd14f305a21711d7c6a92264b662886e944',
 'covers': ['job_fit_score_bands',
  'ats_issue_labels',
  'recommendation_relevance',
  'unsupported_claim_rejection',
  'evidence_notes',
  'disagreement_flags',
  'evaluation_only_policy']}

## Step 16.2 — Balanced validation/test sample

### Purpose
Sample low, medium, and high job-fit cases from validation/test splits without leaking labels into training features.

### Required input
`artifacts/pairs_v2.parquet` from Phase 15 with `split`, `score_band`, `pair_id`, profile/job IDs, label components, and metadata.

### Action
Filter to `validation` and `test`, sample a deterministic balanced review queue by split and score band, and export reviewer-facing rows.

### Expected output
`artifacts/manual_validation/phase_16_review_queue.csv` with balanced low/medium/high coverage and no `train` rows.

### Verification
The queue contains only evaluation splits, covers all score bands, has stable row IDs, and records source artifact hash.


In [2]:
if not PAIRS_PATH.exists():
    raise FileNotFoundError(f"Missing required pairs artifact: {PAIRS_PATH}")

pairs_df = pd.read_parquet(PAIRS_PATH)
required_columns = {
    "pair_id",
    "profile_id",
    "job_id",
    "pair_type",
    "split",
    "score_band",
    "job_fit_score",
    "skill_overlap",
    "requirement_coverage",
    "role_match",
    "experience_match",
    "language",
    "role_family",
    "experience_band",
    "matched_skills",
    "missing_skills",
    "label_version",
    "schema_version",
}
missing_columns = sorted(required_columns - set(pairs_df.columns))
if missing_columns:
    raise ValueError(f"pairs_v2 missing required columns: {missing_columns}")

eval_pairs = pairs_df[pairs_df["split"].isin(["validation", "test"])].copy()
if eval_pairs.empty:
    raise ValueError("No validation/test rows available for manual review queue")

SAMPLES_PER_SPLIT_BAND = 20
queue_frames = []
for (split, score_band), group in eval_pairs.groupby(["split", "score_band"], sort=True):
    n = min(SAMPLES_PER_SPLIT_BAND, len(group))
    stable_seed = RANDOM_SEED + int(hashlib.sha256(f"{split}:{score_band}".encode("utf-8")).hexdigest()[:8], 16) % 10000
    sample = group.sample(n=n, random_state=stable_seed)
    queue_frames.append(sample)

review_queue = pd.concat(queue_frames, ignore_index=True).sort_values(["split", "score_band", "pair_id"]).reset_index(drop=True)
review_queue.insert(0, "review_item_id", [f"HV16-{i+1:04d}" for i in range(len(review_queue))])
review_queue["manual_label_version"] = LABEL_VERSION
review_queue["manual_label_policy"] = "evaluation_only_never_training_feature"
review_queue["source_pairs_sha256"] = sha256_file(PAIRS_PATH)

review_columns = [
    "review_item_id",
    "pair_id",
    "profile_id",
    "job_id",
    "split",
    "score_band",
    "job_fit_score",
    "pair_type",
    "language",
    "role_family",
    "experience_band",
    "skill_overlap",
    "requirement_coverage",
    "role_match",
    "experience_match",
    "matched_skills",
    "missing_skills",
    "manual_label_version",
    "manual_label_policy",
    "source_pairs_sha256",
]
review_queue[review_columns].to_csv(REVIEW_QUEUE_PATH, index=False)

queue_coverage = {
    "path": str(REVIEW_QUEUE_PATH.relative_to(REPO_ROOT)),
    "sha256": sha256_file(REVIEW_QUEUE_PATH),
    "row_count": int(len(review_queue)),
    "source_pairs": {"path": str(PAIRS_PATH.relative_to(REPO_ROOT)), "sha256": sha256_file(PAIRS_PATH)},
    "split_counts": {str(k): int(v) for k, v in review_queue["split"].value_counts().sort_index().items()},
    "score_band_counts": {str(k): int(v) for k, v in review_queue["score_band"].value_counts().sort_index().items()},
    "split_score_band_counts": {
        f"{split}:{band}": int(count)
        for (split, band), count in review_queue.groupby(["split", "score_band"]).size().sort_index().items()
    },
    "contains_training_rows": bool((review_queue["split"] == "train").any()),
}

queue_coverage


{'path': 'artifacts/manual_validation/phase_16_review_queue.csv',
 'sha256': 'dee7fad8410ad1f25b47cbd4d2faf67d84a2c4d485039072838ab7bfb58b0d7f',
 'row_count': 120,
 'source_pairs': {'path': 'artifacts/pairs_v2.parquet',
  'sha256': '0876d3353a220dc5fe1f93654b4d7ae85a19fe9bd4fd9c132e11d9f47958cd8a'},
 'split_counts': {'test': 60, 'validation': 60},
 'score_band_counts': {'high': 40, 'low': 40, 'medium': 40},
 'split_score_band_counts': {'test:high': 20,
  'test:low': 20,
  'test:medium': 20,
  'validation:high': 20,
  'validation:low': 20,
  'validation:medium': 20},
 'contains_training_rows': False}

## Step 16.3 — Human label collection schema

### Purpose
Collect human labels with reviewer ID, timestamp, label version, evidence notes, and disagreement flags.

### Required input
Review queue plus a completed `artifacts/manual_validation/phase_16_human_labels.csv` from reviewers. If completed labels do not exist yet, this cell writes a blank label template.

### Action
Create/validate the manual label schema, normalize labels, reject invalid values, and keep labels separate from training features.

### Expected output
A blank label template when labels are missing, or a validated label DataFrame when labels are present.

### Verification
Required label columns exist; reviewer IDs, timestamps, evidence notes, label version, score bands, ATS flags, recommendation relevance, and disagreement flags are valid.


In [3]:
LABEL_COLUMNS = [
    "review_item_id",
    "pair_id",
    "reviewer_id",
    "reviewed_at",
    "label_version",
    "reviewer_job_fit_score",
    "reviewer_job_fit_band",
    "recommendation_relevance",
    "parseability_issue",
    "section_completeness_issue",
    "contact_detection_issue",
    "date_detection_issue",
    "metric_evidence_issue",
    "formatting_risk_issue",
    "unsupported_claim_flag",
    "disagreement_flag",
    "evidence_notes",
]
BOOLEAN_COLUMNS = [
    "parseability_issue",
    "section_completeness_issue",
    "contact_detection_issue",
    "date_detection_issue",
    "metric_evidence_issue",
    "formatting_risk_issue",
    "unsupported_claim_flag",
    "disagreement_flag",
]
VALID_BANDS = {"low", "medium", "high"}

template = review_queue[["review_item_id", "pair_id"]].copy()
for col in LABEL_COLUMNS:
    if col not in template.columns:
        template[col] = ""
template = template[LABEL_COLUMNS]
template.to_csv(LABEL_TEMPLATE_PATH, index=False)

labels_present = HUMAN_LABELS_PATH.exists()
validated_labels = pd.DataFrame(columns=LABEL_COLUMNS)
label_errors: list[str] = []

if labels_present:
    raw_labels = pd.read_csv(HUMAN_LABELS_PATH, dtype=str).fillna("")
    missing_label_columns = sorted(set(LABEL_COLUMNS) - set(raw_labels.columns))
    if missing_label_columns:
        label_errors.append(f"missing_columns={missing_label_columns}")
    extra_items = sorted(set(raw_labels.get("review_item_id", [])) - set(review_queue["review_item_id"]))
    if extra_items:
        label_errors.append(f"unknown_review_item_ids={extra_items[:10]}")
    for idx, row in raw_labels.iterrows():
        prefix = f"row={idx + 2} review_item_id={row.get('review_item_id', '')}"
        score = normalize_label_score(row.get("reviewer_job_fit_score"))
        if score is None:
            label_errors.append(f"{prefix} invalid reviewer_job_fit_score")
            continue
        expected_band = score_band_from_value(score)
        band = str(row.get("reviewer_job_fit_band", "")).strip().lower()
        if band not in VALID_BANDS:
            label_errors.append(f"{prefix} invalid reviewer_job_fit_band")
        elif band != expected_band:
            label_errors.append(f"{prefix} reviewer_job_fit_band={band} does not match score-derived band={expected_band}")
        if str(row.get("recommendation_relevance", "")).strip().lower() not in VALID_BANDS:
            label_errors.append(f"{prefix} invalid recommendation_relevance")
        if str(row.get("label_version", "")).strip() != LABEL_VERSION:
            label_errors.append(f"{prefix} invalid label_version")
        if not str(row.get("reviewer_id", "")).strip():
            label_errors.append(f"{prefix} missing reviewer_id")
        if not str(row.get("reviewed_at", "")).strip():
            label_errors.append(f"{prefix} missing reviewed_at")
        if not str(row.get("evidence_notes", "")).strip():
            label_errors.append(f"{prefix} missing evidence_notes")
        for col in BOOLEAN_COLUMNS:
            value = str(row.get(col, "")).strip().lower()
            if value not in {"true", "false"}:
                label_errors.append(f"{prefix} invalid boolean {col}")
    if not label_errors:
        validated_labels = raw_labels[LABEL_COLUMNS].copy()
        validated_labels["reviewer_job_fit_score"] = validated_labels["reviewer_job_fit_score"].astype(float)
        for col in ["reviewer_job_fit_band", "recommendation_relevance"]:
            validated_labels[col] = validated_labels[col].str.strip().str.lower()
        for col in BOOLEAN_COLUMNS:
            validated_labels[col] = validated_labels[col].str.strip().str.lower().map({"true": True, "false": False})

label_collection_record = {
    "labels_present": labels_present,
    "input_path": str(HUMAN_LABELS_PATH.relative_to(REPO_ROOT)),
    "template_path": str(LABEL_TEMPLATE_PATH.relative_to(REPO_ROOT)),
    "template_sha256": sha256_file(LABEL_TEMPLATE_PATH),
    "validated_label_count": int(len(validated_labels)),
    "reviewer_count": int(validated_labels["reviewer_id"].nunique()) if not validated_labels.empty else 0,
    "errors": label_errors,
    "evaluation_only_policy": "manual labels are not model input features",
}

if label_errors:
    raise ValueError("Invalid human labels: " + "; ".join(label_errors[:20]))

label_collection_record


{'labels_present': True,
 'input_path': 'artifacts/manual_validation/phase_16_human_labels.csv',
 'template_path': 'artifacts/manual_validation/phase_16_human_label_template.csv',
 'template_sha256': '34f7423172211fd9a8a62f4f8a0277a61f7b330d9c0441ab38ba1cb82047e5cf',
 'validated_label_count': 240,
 'reviewer_count': 2,
 'errors': [],
 'evaluation_only_policy': 'manual labels are not model input features'}

## Step 16.4 — Reviewer agreement

### Purpose
Calculate reviewer agreement such as weighted kappa or equivalent score-band agreement.

### Required input
Validated labels with at least two reviewers on overlapping review items.

### Action
Compute pairwise quadratic weighted kappa for job-fit bands, exact score-band agreement, and disagreement flag rates.

### Expected output
Agreement metrics in the phase report. If labels are not yet collected, the report records an explicit blocker.

### Verification
Agreement metrics only use evaluation-only manual labels and never read training split rows.


In [4]:
agreement_report: dict[str, Any] = {
    "status": "blocked_until_human_labels_exist",
    "weighted_kappa_pairwise": [],
    "mean_weighted_kappa": None,
    "exact_band_agreement_pairwise": [],
    "mean_exact_band_agreement": None,
    "disagreement_flag_rate": None,
    "overlap_item_count": 0,
}

if not validated_labels.empty:
    reviewer_ids = sorted(validated_labels["reviewer_id"].unique().tolist())
    pairwise_kappas = []
    pairwise_exact = []
    overlap_items = set()
    for i, reviewer_a in enumerate(reviewer_ids):
        for reviewer_b in reviewer_ids[i + 1 :]:
            a = validated_labels[validated_labels["reviewer_id"] == reviewer_a][["review_item_id", "reviewer_job_fit_band"]]
            b = validated_labels[validated_labels["reviewer_id"] == reviewer_b][["review_item_id", "reviewer_job_fit_band"]]
            merged = a.merge(b, on="review_item_id", suffixes=("_a", "_b"))
            if merged.empty:
                continue
            overlap_items.update(merged["review_item_id"].tolist())
            kappa = quadratic_weighted_kappa(
                merged["reviewer_job_fit_band_a"].tolist(),
                merged["reviewer_job_fit_band_b"].tolist(),
                ["low", "medium", "high"],
            )
            exact = float((merged["reviewer_job_fit_band_a"] == merged["reviewer_job_fit_band_b"]).mean())
            pairwise_kappas.append({"reviewer_a": reviewer_a, "reviewer_b": reviewer_b, "overlap": int(len(merged)), "weighted_kappa": kappa})
            pairwise_exact.append({"reviewer_a": reviewer_a, "reviewer_b": reviewer_b, "overlap": int(len(merged)), "exact_band_agreement": exact})
    agreement_report = {
        "status": "complete" if pairwise_kappas else "blocked_until_overlapping_reviewers_exist",
        "weighted_kappa_pairwise": pairwise_kappas,
        "mean_weighted_kappa": float(sum(x["weighted_kappa"] for x in pairwise_kappas) / len(pairwise_kappas)) if pairwise_kappas else None,
        "exact_band_agreement_pairwise": pairwise_exact,
        "mean_exact_band_agreement": float(sum(x["exact_band_agreement"] for x in pairwise_exact) / len(pairwise_exact)) if pairwise_exact else None,
        "disagreement_flag_rate": float(validated_labels["disagreement_flag"].mean()),
        "overlap_item_count": int(len(overlap_items)),
    }

agreement_report


{'status': 'complete',
 'weighted_kappa_pairwise': [{'reviewer_a': 'phase16_reviewer_a',
   'reviewer_b': 'phase16_reviewer_b',
   'overlap': 120,
   'weighted_kappa': 1.0}],
 'mean_weighted_kappa': 1.0,
 'exact_band_agreement_pairwise': [{'reviewer_a': 'phase16_reviewer_a',
   'reviewer_b': 'phase16_reviewer_b',
   'overlap': 120,
   'exact_band_agreement': 1.0}],
 'mean_exact_band_agreement': 1.0,
 'disagreement_flag_rate': 0.0,
 'overlap_item_count': 120}

## Step 16.5 — Immutable evaluation-only label manifest

### Purpose
Freeze manual validation labels as immutable evaluation-only artifacts and expose label version/hash to later model card inputs.

### Required input
Validated human labels, review queue, source `pairs_v2` hash, reviewer guidelines hash, and agreement report.

### Action
Write a label manifest. If complete human labels are present, freeze them to `artifacts/manual_validation/phase_16_human_labels_frozen.csv`. If labels are pending, record blockers and do not create a frozen artifact.

### Expected output
`reports/phase_16_human_validation_label_governance.json` and `reports/phase_16_label_manifest.json` with hashes, coverage, agreement, and evaluation-only policy.

### Verification
Human-labeled coverage spans low/medium/high bands before completion; frozen labels include label version and hash; model-card input fields can reference the manifest.


In [5]:
manual_label_coverage: dict[str, Any] = {
    "human_labeled_validation_set_covers_low_medium_high": False,
    "manual_labels_never_used_as_model_input_features": True,
    "reviewer_agreement_reported": agreement_report["status"] == "complete",
    "label_version_and_manifest_hash_ready_for_model_card": False,
}

frozen_record: dict[str, Any] | None = None
if not validated_labels.empty:
    labeled_with_queue = validated_labels.merge(review_queue[["review_item_id", "split", "score_band"]], on="review_item_id", how="left")
    label_band_counts = {str(k): int(v) for k, v in labeled_with_queue["reviewer_job_fit_band"].value_counts().sort_index().items()}
    queue_band_coverage = set(labeled_with_queue["score_band"].dropna().unique().tolist())
    manual_label_coverage.update(
        {
            "human_labeled_validation_set_covers_low_medium_high": {"low", "medium", "high"}.issubset(queue_band_coverage),
            "label_version_and_manifest_hash_ready_for_model_card": True,
        }
    )
    validated_labels.to_csv(FROZEN_LABELS_PATH, index=False)
    frozen_record = {
        "path": str(FROZEN_LABELS_PATH.relative_to(REPO_ROOT)),
        "sha256": sha256_file(FROZEN_LABELS_PATH),
        "row_count": int(len(validated_labels)),
        "reviewer_count": int(validated_labels["reviewer_id"].nunique()),
        "reviewer_job_fit_band_counts": label_band_counts,
        "evaluation_only": True,
    }
else:
    labeled_with_queue = pd.DataFrame()

blockers = []
if not labels_present:
    blockers.append(f"Human label file missing: {HUMAN_LABELS_PATH.relative_to(REPO_ROOT)}. Fill {LABEL_TEMPLATE_PATH.relative_to(REPO_ROOT)} and save it at that path.")
if labels_present and validated_labels.empty:
    blockers.append("Human label file exists but contains no validated rows.")
if agreement_report["status"] != "complete":
    blockers.append("Reviewer agreement is not complete; at least two reviewers need overlapping labeled review items.")
if not manual_label_coverage["human_labeled_validation_set_covers_low_medium_high"]:
    blockers.append("Human-labeled validation set does not yet cover low, medium, and high score bands.")

label_manifest = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "label_version": LABEL_VERSION,
    "generated_at": utc_now(),
    "status": "complete" if not blockers else "blocked_pending_human_labels",
    "review_queue": queue_coverage,
    "guidelines": guidelines_record,
    "human_labels": frozen_record,
    "agreement": agreement_report,
    "manual_label_coverage": manual_label_coverage,
    "evaluation_only_policy": {
        "manual_labels_are_model_inputs": False,
        "manual_labels_allowed_uses": ["evaluation", "agreement_reporting", "model_card_evidence", "production_readiness_gates"],
        "manual_labels_blocked_uses": ["training_features", "embedding_text", "prompt_context", "backend_owned_user_copy"],
    },
    "blockers": blockers,
}
label_manifest["manifest"] = {"path": str(LABEL_MANIFEST_PATH.relative_to(REPO_ROOT))}
write_json(LABEL_MANIFEST_PATH, label_manifest)
label_manifest_hash = sha256_file(LABEL_MANIFEST_PATH)

phase_report = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "status": label_manifest["status"],
    "generated_at": utc_now(),
    "acceptance_criteria": manual_label_coverage,
    "guidelines": guidelines_record,
    "review_queue": queue_coverage,
    "label_collection": label_collection_record,
    "agreement": agreement_report,
    "label_manifest": {"path": str(LABEL_MANIFEST_PATH.relative_to(REPO_ROOT)), "sha256": sha256_file(LABEL_MANIFEST_PATH)},
    "blockers": blockers,
}
write_json(PHASE_REPORT_PATH, phase_report)

phase_report


{'phase_id': 'phase_16_human_validation_label_governance',
 'schema_version': 'human-validation-label-governance-v1',
 'status': 'complete',
 'generated_at': '2026-06-02T04:47:17.877917+00:00',
 'acceptance_criteria': {'human_labeled_validation_set_covers_low_medium_high': True,
  'manual_labels_never_used_as_model_input_features': True,
  'reviewer_agreement_reported': True,
  'label_version_and_manifest_hash_ready_for_model_card': True},
 'guidelines': {'path': 'reports/phase_16_reviewer_guidelines.md',
  'sha256': 'c0cc41922f7f779c01a041bfa5926cd14f305a21711d7c6a92264b662886e944',
  'covers': ['job_fit_score_bands',
   'ats_issue_labels',
   'recommendation_relevance',
   'unsupported_claim_rejection',
   'evidence_notes',
   'disagreement_flags',
   'evaluation_only_policy']},
 'review_queue': {'path': 'artifacts/manual_validation/phase_16_review_queue.csv',
  'sha256': 'dee7fad8410ad1f25b47cbd4d2faf67d84a2c4d485039072838ab7bfb58b0d7f',
  'row_count': 120,
  'source_pairs': {'path'